# 06 — Renforcer l'usage des mots-clés

Le modèle pré-entraîné du notebook 05 écrit des histoires cohérentes, mais
il lit mal la consigne :

| | taux de reprise | référence (consigne d'un autre texte) |
|---|---|---|
| histoires | 42,7 % | 21,7 % |

La référence est haute : un mot-clé sur cinq apparaîtrait **même si le
modèle ignorait la consigne**. Les mots-clés étaient tirés uniformément
parmi les mots porteurs de sens, et beaucoup sont présents dans presque
toutes les histoires : `time` (*Once upon a time*), `day`, `happy`,
`little`… Pour ces mots, lire la consigne ne sert à rien : le modèle les
écrit de toute façon. L'entraînement ne le pousse donc pas assez à
regarder la consigne.

L'idée de ce notebook : **tirer plus souvent les mots rares**. Pour prédire
`moon` ou `pie` dans une histoire, le seul indice est la consigne. On
reprend le modèle du notebook 05 et on continue l'entraînement ~3 000 pas
avec ce nouveau tirage, puis on compare avant / après.

In [1]:
import copy
import json
import math
import random
import shutil
import time
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, RandomSampler
from torch.utils.tensorboard import SummaryWriter

from comptines import (BPETokenizer, GPTLanguageModel, TokenTextDataset, keyword_hits, sample_keywords,
                       write_text, write_texts)

In [2]:
SEED = 1337
torch.manual_seed(SEED)
random.seed(SEED)

try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

BPE_DIR = ROOT / "data" / "bpe"
RUNS_DIR = ROOT / "runs" / "06_mots_cles"
PRETRAINED_PATH = ROOT / "checkpoints" / "gpt_bpe_preentraine.pt"
CHECKPOINT_PATH = ROOT / "checkpoints" / "gpt_bpe_mots_cles.pt"

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"device : {device}")

device : cuda


## 1. La rareté d'un mot : l'IDF

L'**IDF** (*inverse document frequency*) vient de la recherche
d'information : un mot présent dans peu de documents est plus informatif
qu'un mot présent partout. Avec `N` histoires, dont `df(w)` contiennent le
mot `w` :

$$\text{idf}(w) = \ln\frac{1 + N}{1 + \text{df}(w)} + 1$$

C'est la formule de scikit-learn (`TfidfVectorizer`) : le `+ 1` final
garantit un poids d'au moins 1, même pour un mot présent partout. Les mots
porteurs de sens de chaque histoire ont été calculés au notebook 04 (champ
`keywords`) : il suffit de compter dans combien d'histoires apparaît chacun.

In [3]:
tokenizer = BPETokenizer(json.loads((BPE_DIR / "vocab.json").read_text()))
train_packed = torch.load(BPE_DIR / "tinystories_train.pt")
val_packed = torch.load(BPE_DIR / "tinystories_val.pt")
rhymes_val_packed = torch.load(BPE_DIR / "comptines_val.pt")

n_docs = len(train_packed["keywords"])
df = Counter(w for words in train_packed["keywords"] for w in words)  # content_words est déjà sans doublon
idf = {w: math.log((1 + n_docs) / (1 + n)) + 1 for w, n in df.items()}

print(f"{len(idf):,} mots   idf de {min(idf.values()):.2f} à {max(idf.values()):.2f}\n")
print("les plus fréquents :", ", ".join(f"{w} ({df[w] / n_docs:.0%}, idf {idf[w]:.1f})" for w, _ in df.most_common(8)))
for w in ["moon", "pie", "garden", "king", "cat", "star"]:
    print(f"  {w:7} dans {df[w] / n_docs:6.2%} des histoires   idf {idf[w]:.1f}")

10,257 mots   idf de 1.07 à 11.22

les plus fréquents : day (93%, idf 1.1), time (72%, idf 1.3), happy (69%, idf 1.4), big (69%, idf 1.4), named (67%, idf 1.4), said (66%, idf 1.4), saw (62%, idf 1.5), little (60%, idf 1.5)
  moon    dans  0.36% des histoires   idf 6.6
  pie     dans  0.16% des histoires   idf 7.4
  garden  dans  3.97% des histoires   idf 4.2
  king    dans  0.50% des histoires   idf 6.3
  cat     dans 14.76% des histoires   idf 2.9
  star    dans  0.46% des histoires   idf 6.4


`time` est dans presque toutes les histoires : son poids est proche de 1.
`moon` ou `pie` pèsent 4 à 6 fois plus. Avec `sample_keywords(..., idf=idf)`
(module `comptines.py`), la chance d'un mot d'être tiré est proportionnelle à
son IDF. Comparons les deux tirages sur une même histoire :

In [4]:
story_words = train_packed["keywords"][3]
rng = random.Random(SEED)
uniform = Counter(w for _ in range(2_000) for w in sample_keywords(story_words, rng))
weighted = Counter(w for _ in range(2_000) for w in sample_keywords(story_words, rng, idf=idf))
print(f"{'mot':12} {'idf':>5} {'uniforme':>9} {'pondéré':>8}")
for w in sorted(story_words, key=idf.get)[:5] + sorted(story_words, key=idf.get)[-5:]:
    print(f"{w:12} {idf[w]:>5.1f} {uniform[w]:>9} {weighted[w]:>8}")

mot            idf  uniforme  pondéré
day            1.1        99       22
time           1.3        88       31
big            1.4        87       28
little         1.5        83       33
friends        1.9        92       35
leaned         7.1        94      135
thumb          8.1        79      179
reminder       8.2        92      181
curiously      8.3        93      174
appreciation   9.0        96      191


Les mots génériques (`time`, `day`) sortent beaucoup moins souvent, les mots
spécifiques à cette histoire beaucoup plus. Le tirage reste aléatoire : un
mot courant peut encore sortir, et le modèle continue d'apprendre à les
utiliser aussi.

## 2. Deux mesures, avant et après

On mesure le taux de reprise (et sa référence) avec des consignes **fixes**,
les mêmes pour le modèle avant et après :

- **uniforme** : les consignes du notebook 05, pour comparer à ses chiffres ;
- **pondéré** : des consignes tirées avec l'IDF, celles que le modèle verra
  maintenant à l'entraînement, et plus proches de ce qu'un utilisateur
  donnerait (`cat, moon` plutôt que `time, day`) ;
- **comptines** (pondéré) : les 299 comptines de validation, jamais vues.

In [5]:
block_size = 256
batch_size = 32
val_uniform = TokenTextDataset(val_packed, tokenizer, block_size, fixed_keywords=True)
val_weighted = TokenTextDataset(val_packed, tokenizer, block_size, fixed_keywords=True, idf=idf)
rhymes_val = TokenTextDataset(rhymes_val_packed, tokenizer, block_size, fixed_keywords=True, idf=idf)
eval_sets = {"histoires, uniforme": val_uniform, "histoires, pondéré": val_weighted, "comptines, pondéré": rhymes_val}
n_eval = 300


def evaluate_keywords(model: nn.Module) -> dict[str, tuple[float, float]]:
    """Taux de reprise et référence, pour chaque jeu de consignes fixes."""
    scores = {}
    for name, dataset in eval_sets.items():
        keyword_sets = [k for k in (dataset.keywords_of(i) for i in range(min(n_eval, len(dataset)))) if k]
        torch.manual_seed(SEED)
        generated = write_texts(model, tokenizer, keyword_sets)
        hits = sum(keyword_hits(k, g) for k, g in zip(keyword_sets, generated)) / len(generated)
        shifted = generated[1:] + generated[:1]  # le texte écrit pour une AUTRE consigne
        baseline = sum(keyword_hits(k, g) for k, g in zip(keyword_sets, shifted)) / len(generated)
        scores[name] = (hits, baseline)
    return scores


def print_scores(scores: dict[str, tuple[float, float]]) -> None:
    for name, (hits, baseline) in scores.items():
        print(f"  {name:20} taux de reprise {hits:6.1%}   référence {baseline:5.1%}")

## 3. Le modèle du notebook 05, avant

In [6]:
checkpoint = torch.load(PRETRAINED_PATH, map_location=device)
config = checkpoint["config"]
model = GPTLanguageModel(**config).to(device)
model.load_state_dict(checkpoint["model_state"])
print(f"modèle rechargé : {sum(p.numel() for p in model.parameters()):,} paramètres   val {checkpoint['val_loss']:.4f}\n")

t0 = time.perf_counter()
scores_before = evaluate_keywords(model)
print_scores(scores_before)
print(f"\n({time.perf_counter() - t0:.0f} s de génération)")

modèle rechargé : 8,429,354 paramètres   val 1.5787



  histoires, uniforme  taux de reprise  40.6%   référence 23.8%
  histoires, pondéré   taux de reprise  29.6%   référence 12.6%
  comptines, pondéré   taux de reprise  14.0%   référence  1.8%

(455 s de génération)


## 4. Continuer l'entraînement, avec le tirage pondéré

On ne repart pas de zéro : on **continue** l'entraînement du modèle du
notebook 05 (*continued pretraining*). Il sait déjà écrire ; on ne veut que
changer la consigne qu'il voit. D'où un *learning rate* plus petit que celui
du pré-entraînement (5e-4 au plus au lieu de 1e-3), avec un court
échauffement et une décroissance en cosinus : de trop grands pas
abîmeraient ce qui est déjà appris.

La boucle est celle du notebook 05 (`bfloat16`, meilleur modèle de
validation gardé). La validation utilise les consignes pondérées.

In [7]:
train_dataset = TokenTextDataset(train_packed, tokenizer, block_size, idf=idf)

max_steps = 3_000
warmup_steps = 100
max_lr = 5e-4
min_lr = 5e-5
eval_every = 500
val_batches = 25


def lr_at(step: int) -> float:
    """Learning rate au pas `step` : montée linéaire, puis cosinus de `max_lr` à `min_lr`."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def make_loader(dataset: TokenTextDataset, n_batches: int) -> DataLoader:
    """Un `DataLoader` qui tire `n_batches` lots au hasard, avec remise."""
    sampler = RandomSampler(dataset, replacement=True, num_samples=n_batches * batch_size)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler)


def autocast():
    """Calcul en bfloat16 sur GPU, rien de spécial sur CPU."""
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()


val_loader = make_loader(val_weighted, val_batches)

In [8]:
def train_step(model: nn.Module, optimizer: torch.optim.Optimizer, X: torch.Tensor, Y: torch.Tensor) -> float:
    """Un pas d'entraînement sur un lot ; renvoie la loss du lot."""
    model.train()  # dropout actif
    with autocast():
        _, loss = model(X.to(device), Y.to(device))  # 1. forward, loss comprise
    optimizer.zero_grad(set_to_none=True)
    loss.backward()  # 2. backward
    optimizer.step()  # 3. step
    return loss.item()


@torch.no_grad()
def test(model: nn.Module, loader: DataLoader) -> float:
    """Mesure la loss moyenne sur les lots du loader, sans apprendre."""
    model.eval()  # dropout coupé
    total = 0.0
    for X, Y in loader:
        with autocast():
            _, loss = model(X.to(device), Y.to(device))
        total += loss.item()
    return total / len(loader)


def run_training(model: nn.Module, run: str, max_steps: int) -> dict:
    """Entraîne pendant `max_steps` pas, mesure tous les `eval_every` pas,
    et recharge à la fin les poids du meilleur pas de validation."""
    torch.manual_seed(SEED)
    log_dir = RUNS_DIR / run
    shutil.rmtree(log_dir, ignore_errors=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)
    train_losses, history = [], []
    best_val, best_step, best_state = test(model, val_loader), 0, copy.deepcopy(model.state_dict())
    print(f"pas      0   val {best_val:.3f}   (point de départ)")
    start = time.perf_counter()

    with SummaryWriter(log_dir=str(log_dir)) as writer:
        for step, (X, Y) in enumerate(make_loader(train_dataset, max_steps), start=1):
            for group in optimizer.param_groups:
                group["lr"] = lr_at(step - 1)
            train_losses.append(train_step(model, optimizer, X, Y))

            if step % eval_every == 0 or step == max_steps:
                train_loss = sum(train_losses) / len(train_losses)
                val_loss = test(model, val_loader)
                writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, step)
                history.append((step, train_loss, val_loss))
                print(f"pas {step:>6,}   train {train_loss:.3f}   val {val_loss:.3f}   {time.perf_counter() - start:>5.0f} s")
                train_losses = []
                if val_loss < best_val:
                    best_val, best_step, best_state = val_loss, step, copy.deepcopy(model.state_dict())

    duration = time.perf_counter() - start
    model.load_state_dict(best_state)
    print(f"\n[{run}] {max_steps:,} pas   {duration / 60:.0f} min   meilleure val {best_val:.4f} au pas {best_step:,}")
    return {"history": history, "best_step": best_step, "best_val": best_val, "duration": duration}

Le point de départ de la validation est mesuré **avant** le premier pas :
c'est la loss du modèle du notebook 05 sur les consignes pondérées. Si elle
baisse, le modèle utilise mieux les mots rares de la consigne pour prédire
le texte.

In [9]:
result = run_training(model, "idf", max_steps)

pas      0   val 1.593   (point de départ)


pas    500   train 1.561   val 1.633     135 s


pas  1,000   train 1.568   val 1.642     274 s


pas  1,500   train 1.532   val 1.608     411 s


pas  2,000   train 1.510   val 1.600     556 s


pas  2,500   train 1.481   val 1.569     699 s


pas  3,000   train 1.461   val 1.592     840 s

[idf] 3,000 pas   14 min   meilleure val 1.5687 au pas 2,500


## 5. Après

In [10]:
scores_after = evaluate_keywords(model)
print(f"{'':20} {'avant':>22}   {'après':>22}")
for name in eval_sets:
    (h0, b0), (h1, b1) = scores_before[name], scores_after[name]
    print(f"{name:20} {h0:7.1%} (réf. {b0:5.1%})   {h1:7.1%} (réf. {b1:5.1%})")

                                      avant                    après
histoires, uniforme    40.6% (réf. 23.8%)     44.1% (réf. 21.9%)
histoires, pondéré     29.6% (réf. 12.6%)     34.4% (réf. 12.4%)
comptines, pondéré     14.0% (réf.  1.8%)     15.3% (réf.  2.6%)


La bonne lecture est l'**écart** entre le taux de reprise et la référence :
c'est la part des mots-clés que le modèle reprend *parce qu'ils sont dans la
consigne*.

Les consignes de démonstration, avant le fine-tuning sur les comptines (le
modèle écrit encore des histoires, c'est normal) :

In [11]:
torch.manual_seed(SEED)
for words in [["cat", "moon"], ["king", "pie"], ["rain", "garden"], ["dog", "bone", "star"], ["girl", "ball", "happy"]]:
    text = write_text(model, tokenizer, words)
    found = [w for w in words if keyword_hits([w], text)]
    print(f"--- {', '.join(words)}   (repris : {', '.join(found) or 'aucun'}) ---")
    print(text, end="\n\n")

--- cat, moon   (repris : moon) ---
Once upon a time, there was a kind man named Tom. He had a big house and many toys. Tom liked to play with his toys all day long. One day, he found a big, red ball in his room. He thought it would be fun to play with.
Tom took the ball outside and started to play with it. The ball went up, up, up, up into the sky. Tom did not see a big, bright light. He was scared at first, but then he remembered it was his play. He ran back to his house and ran back to his house.
When Tom got there, he saw that the light was gone. He was very sad and scared. But then, he saw his ball near the moon. He did not know what to do. He went to his mom and said, "Mom, I found a ball!" His mom saw the ball and smiled. She said, "This is for now, Tom. You can play with the ball too." Tom was happy again and played with his ball all day long.



--- king, pie   (repris : pie) ---
Once upon a time, there was a little girl named Lily. She was a brave girl who loved to play with her friends. One day, Lily saw a pie on the table.
Lily's friend, Tim, came over to play. Tim saw the pie and wanted some too. "Can I taste that pie?" Lily asked. Tim said, "No, I can't taste that pie! It is not ours."
Lily and Tim played together with the pie. They took turns tasting and laughing. They laughed as they ate the pie. At the end of the day, Lily and Tim were very happy. They knew they would always be best friends and always share their yummy pie with each other.



--- rain, garden   (repris : aucun) ---
Once upon a time, there was a pink cat. The cat lived in a house with a sink. The cat liked to play with the bubbles.
One day, the cat saw a new soap in the sink. The soap was pink and pink. The cat wanted to play with the soap. So, the cat went to the sink and started to play.
The cat played with the soap all day. The soap made the pink water all day long. The cat was very happy. And from that day on, the cat and the soap became best friends. They played together every day, and the pink cat always was a pink cat.



--- dog, bone, star   (repris : dog) ---
Once upon a time, there was a little dog named Spot. Spot was a mild dog. He had a soft head and a warm ball. Spot loved to play with his ball every day.
One day, Spot saw a big ball. He wanted to play with it. Spot ran fast to the ball. He ran very fast. At the end, Spot caught the ball with his mouth.
Spot played with the ball all day. He was very happy. When the sun went down, Spot went home. He told his mom and dad about the big ball. They all wanted to play with the ball too. And they all played happily ever after.



--- girl, ball, happy   (repris : girl, ball) ---
Once upon a time, there was a little girl named Lily. She had a ball. Lily loved to play with her ball. One day, she saw a big ball in the park. She ran to the ball and began to play with it.
Lily's mom saw her playing with the big ball. She said, "Lily, be careful! Don't touch your ball." But Lily did not listen. She wanted to get the ball and play with it.
Lily and her mom played with the big ball. They had so much fun. But then, they saw a loud noise. Lily was scared. She ran to her mom and said, "Mom, the big ball is coming! I was stuck in a hole!" Her mom hugged her and said, "Don't worry, Lily. We will find a new way to get the ball out." Lily and her mom looked and found a new ball. Lily learned to listen to her mom's advice.



## 6. Sauvegarder

Le modèle, et la table d'IDF : le notebook 07 tire les mots-clés des
comptines avec les mêmes poids.

In [12]:
(BPE_DIR / "idf.json").write_text(json.dumps(idf))
torch.save({
    "model_state": model.state_dict(),
    "config": config,
    "gpt2_ids": tokenizer.gpt2_ids,
    "val_loss": result["best_val"],
    "history": result["history"],
    "keyword_scores": {"avant": scores_before, "après": scores_after},
}, CHECKPOINT_PATH)
print(f"→ {CHECKPOINT_PATH.relative_to(ROOT)}   → {(BPE_DIR / 'idf.json').relative_to(ROOT)}")

→ checkpoints\gpt_bpe_mots_cles.pt   → data\bpe\idf.json
